# XGBoost ablation confirmation: ratios and charging helpers

This notebook confirms the small gain observed when removing ratios and charging helpers. The completed `xgboost_ablation_study.ipynb`, its screening outputs, and all files in `XGBoost/baseline/` and `XGBoost/advanced/` remain untouched.

The candidate removes the same 12 ratio/charging helper columns tested in screening; the reference keeps them. Raw income, commute, and station counts remain in both configurations. All other settings, class weights, outer folds, inner encoding folds, and model seeds stay identical. The full reference is the strongest **single recipe-residual model**, not the final three-model leaderboard blend. Parameters are frozen in the accompanying snapshot from your successful run.

No Optuna search or test submission is performed during the study. The output is evidence for choosing the next improvement.

- **Configured confirmation:** full data, 10 folds, 2 split seeds, 2 configurations = 40 fits.
- Compare `full_reference` against `no_ratios_charging` using paired OOF AUC differences.
- Screening found a gain of approximately 0.000029 AUC. A positive result across both split seeds supports the candidate, but does not guarantee a leaderboard improvement.
- Supervised encodings are cross-fitted inside each outer training fold.
- Finished fits are checkpointed; interrupted runs resume without reusing incompatible results.
- Keep raw columns when removing engineered groups. A group-removal result measures its added value **given the remaining features**, not its causal or standalone importance.


## 1. Settings

The confirmation shortlist is already configured. Select your existing `venv` kernel, save this notebook, then restart the kernel and run all cells in order. Defaults are CPU-safe and use the full dataset. This run has 40 fits and can take several hours.

Set `DEVICE = 'cuda'` only with a working NVIDIA/CUDA setup. Keep `MAX_BOOST_ROUNDS = 6000` for the initial comparison; results report whether training reaches that limit. Use a larger common budget for all experiments if necessary.

Save the notebook before running it. The output fingerprint hashes its saved code, inputs, parameters, library versions, and experiment definitions. Changing settings creates a separate output directory. No historical result file is deleted.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost as xgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

PIPELINE_VERSION = '1.0.0'
GLOBAL_SEED = 20260915
MODE = 'confirm'                 # Confirm the small improvement seen in screening
CV_SEEDS = None                   # None chooses reproducible mode-specific seeds
SELECTED_EXPERIMENTS = [
    'full_reference',
    'no_ratios_charging',
]
# Keep this shortlist fixed: 2 configurations x 10 folds x 2 split seeds = 40 fits.
INNER_TE_FOLDS = 5
MAX_BOOST_ROUNDS = 6000
EARLY_STOPPING_ROUNDS = 200
MAX_BIN = 512
DEVICE = 'cpu'
NTHREAD = -1
RESUME_COMPLETED_FOLDS = True

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 100)

if MODE not in {'screen', 'confirm'}:
    raise ValueError("MODE must be 'screen' or 'confirm'.")
N_SPLITS = 3 if MODE == 'screen' else 10
SPLIT_SEEDS = tuple(CV_SEEDS) if CV_SEEDS is not None else (
    (GLOBAL_SEED,) if MODE == 'screen' else (GLOBAL_SEED + 17, GLOBAL_SEED + 1017)
)
assert SPLIT_SEEDS and len(set(SPLIT_SEEDS)) == len(SPLIT_SEEDS)
assert all(isinstance(seed, int) and 0 <= seed < 2**32 for seed in SPLIT_SEEDS)
assert INNER_TE_FOLDS >= 2 and MAX_BOOST_ROUNDS > EARLY_STOPPING_ROUNDS > 0
assert MAX_BIN >= 2
assert SELECTED_EXPERIMENTS[0] == 'full_reference', 'Keep full_reference first.'
assert len(set(SELECTED_EXPERIMENTS)) == len(SELECTED_EXPERIMENTS)

VERSIONS = {
    'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
    'sklearn': sklearn.__version__, 'xgboost': xgb.__version__,
}
print('Versions:', VERSIONS)
print(f'{MODE}: {N_SPLITS} folds x {len(SPLIT_SEEDS)} split seeds x '
      f'{len(SELECTED_EXPERIMENTS)} experiments')


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'playground-series-s6e9' / 'train.csv').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter in this repository or in the XGBoost/ablation folder.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
STUDY_DIR = PROJECT_ROOT / 'XGBoost/ablation'
NOTEBOOK_PATH = STUDY_DIR / 'xgboost_ablation_confirmation.ipynb'
SNAPSHOT_PATH = STUDY_DIR / 'reference_run_snapshot.json'
DATA_DIR = PROJECT_ROOT / 'playground-series-s6e9'
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
TARGET = 'Will_Buy_EV'
ID_COL = 'id'

with SNAPSHOT_PATH.open(encoding='utf-8') as handle:
    reference_snapshot = json.load(handle)
REFERENCE_SEARCH_PARAMS = dict(reference_snapshot['reference_search_params'])
# The strongest historical recipe-residual member used weight 1.0, not 1.4794.
assert REFERENCE_SEARCH_PARAMS['scale_pos_weight'] == 1.0
assert reference_snapshot['run_metadata']['global_seed'] == GLOBAL_SEED

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
assert TARGET in train.columns and TARGET not in test.columns
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert train[TARGET].notna().all()
assert set(train[TARGET].unique()) == {'No', 'Yes'}
assert not set(train[ID_COL]).intersection(test[ID_COL])

y = train[TARGET].map({'No': 0, 'Yes': 1}).astype(np.uint8).to_numpy()
RAW_FEATURES = [column for column in test.columns if column != ID_COL]
assert RAW_FEATURES == [column for column in train.columns if column not in (ID_COL, TARGET)]
assert np.bincount(y).min() >= max(N_SPLITS, INNER_TE_FOLDS) * 2

print(f'Train: {train.shape}; test: {test.shape}; positive rate: {y.mean():.6f}')
print('Frozen model parameters:', json.dumps(REFERENCE_SEARCH_PARAMS, indent=2))


## 2. Recreate the reference features exactly

The following feature builder is copied from the successful notebook. Its complete code is embedded here: we never execute or import the old notebook.

The same train+test vocabulary/frequency maps are retained for a faithful label-free, transductive comparison. Medians and winsorization thresholds use only training features. These are unsupervised; all transformations that use targets are handled separately inside folds.

The raw numeric columns are always preserved. Feature groups are registered explicitly in the next cell, so an ablation cannot accidentally drop an unrelated column.


In [ ]:
NUMERIC_COLS = [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
    'Environmental_Concern_Level',
]
CATEGORICAL_COLS = [
    'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
    'Subsidy_Available', 'Range_Anxiety_Level',
]
assert set(NUMERIC_COLS + CATEGORICAL_COLS) == set(RAW_FEATURES)

train_medians = train[NUMERIC_COLS].median()
clip_bounds = {
    col: tuple(train[col].quantile([0.005, 0.995]).astype(float))
    for col in ('Annual_Income_USD', 'Daily_Commute_km')
}


def clean_raw(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[RAW_FEATURES].copy()
    for col in NUMERIC_COLS:
        out[col] = pd.to_numeric(out[col], errors='coerce').fillna(train_medians[col])
    for col in CATEGORICAL_COLS:
        out[col] = out[col].astype('string').fillna('__MISSING__').astype(str)
    return out


train_raw = clean_raw(train)
test_raw = clean_raw(test)
all_raw = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)

frequency_maps = {
    col: all_raw[col].value_counts(dropna=False, normalize=True)
    for col in RAW_FEATURES
}


def ev_recipe_score(frame: pd.DataFrame) -> np.ndarray:
    subsidy = frame['Subsidy_Available'].eq('Yes').to_numpy(dtype=np.float64)
    anxiety_penalty = frame['Range_Anxiety_Level'].map(
        {'Low': 0.0, 'Medium': -1.0, 'High': -3.0}
    ).fillna(-1.0).to_numpy(dtype=np.float64)
    return (
        1.2 * frame['Annual_Income_USD'].to_numpy(dtype=np.float64) / 100_000.0
        + 0.6 * frame['Environmental_Concern_Level'].to_numpy(dtype=np.float64)
        + 2.0 * subsidy
        + anxiety_penalty
    )


def ev_recipe_probability(frame: pd.DataFrame) -> np.ndarray:
    # The source rule is a probit-style threshold with unit Gaussian noise.
    return np.clip(ndtr(ev_recipe_score(frame) - 5.5), 1e-6, 1.0 - 1e-6)


def ev_recipe_logit_margin(frame: pd.DataFrame) -> np.ndarray:
    probability = ev_recipe_probability(frame)
    return np.log(probability / (1.0 - probability)).astype(np.float32)


def safe_divide(numerator: pd.Series, denominator: pd.Series, offset: float = 1.0) -> np.ndarray:
    return numerator.to_numpy(dtype=np.float64) / (denominator.to_numpy(dtype=np.float64) + offset)


def sanitize_feature_names(columns: list[str]) -> list[str]:
    counts: dict[str, int] = {}
    cleaned: list[str] = []
    for column in columns:
        base = re.sub(r'[^0-9A-Za-z_]+', '_', str(column)).strip('_') or 'feature'
        occurrence = counts.get(base, 0)
        counts[base] = occurrence + 1
        cleaned.append(base if occurrence == 0 else f'{base}_{occurrence}')
    return cleaned


def build_base_features(frame: pd.DataFrame) -> pd.DataFrame:
    x = frame[NUMERIC_COLS].astype(np.float32).copy()
    income = frame['Annual_Income_USD']
    commute = frame['Daily_Commute_km']
    cars = frame['Number_of_Cars_Owned']
    home_stations = frame['Charging_Stations_Near_Home']
    work_stations = frame['Charging_Stations_Near_Work']
    concern = frame['Environmental_Concern_Level']
    subsidy = frame['Subsidy_Available'].eq('Yes').astype(np.float32)
    home_charge = frame['Home_Charging_Possible'].eq('Yes').astype(np.float32)
    anxiety = frame['Range_Anxiety_Level'].map({'Low': 0.0, 'Medium': 1.0, 'High': 2.0}).fillna(1.0)

    # Known ordinal/binary meanings; nominal categories are one-hot encoded below.
    x['Subsidy_flag'] = subsidy
    x['Home_charging_flag'] = home_charge
    x['Range_anxiety_ordinal'] = anxiety.astype(np.float32)

    # Charging convenience and capacity interactions.
    total_stations = home_stations + work_stations
    x['Total_charging_stations'] = total_stations.astype(np.float32)
    x['Station_difference'] = (home_stations - work_stations).astype(np.float32)
    x['Station_minimum'] = np.minimum(home_stations, work_stations).astype(np.float32)
    x['Station_maximum'] = np.maximum(home_stations, work_stations).astype(np.float32)
    x['Station_product'] = (home_stations * work_stations).astype(np.float32)
    x['No_nearby_station'] = total_stations.eq(0).astype(np.float32)
    x['Either_station_zero'] = (home_stations.eq(0) | work_stations.eq(0)).astype(np.float32)
    x['Home_access_score'] = (home_stations + 3.0 * home_charge).astype(np.float32)
    x['Commute_per_station'] = safe_divide(commute, total_stations, offset=1.0).astype(np.float32)
    x['Stations_per_commute'] = safe_divide(total_stations, commute, offset=1.0).astype(np.float32)

    # Affordability / usage ratios and monotone companion transforms.
    x['Log_income'] = np.log1p(income).astype(np.float32)
    x['Log_commute'] = np.log1p(commute).astype(np.float32)
    x['Income_per_commute'] = safe_divide(income, commute, offset=1.0).astype(np.float32)
    x['Income_per_car'] = safe_divide(income, cars, offset=0.0).astype(np.float32)
    x['Income_x_subsidy'] = (income * subsidy).astype(np.float32)
    x['Concern_x_subsidy'] = (concern * subsidy).astype(np.float32)
    x['Concern_minus_anxiety'] = (concern - anxiety).astype(np.float32)
    x['Commute_x_anxiety'] = (commute * (anxiety + 1.0)).astype(np.float32)

    # The raw source-data formula gives trees a one-split summary and a strong prior.
    recipe_score = ev_recipe_score(frame)
    x['EV_recipe_score'] = recipe_score.astype(np.float32)
    x['EV_recipe_distance'] = (recipe_score - 5.5).astype(np.float32)
    x['EV_recipe_probability'] = ev_recipe_probability(frame).astype(np.float32)

    # Preserve raw tails. These companion views/flags expose clipping and hard edges.
    for col, (lower, upper) in clip_bounds.items():
        x[f'{col}_winsorized'] = frame[col].clip(lower, upper).astype(np.float32)
    x['Income_at_floor'] = income.eq(30_000).astype(np.float32)
    x['Commute_at_floor'] = commute.eq(5.0).astype(np.float32)
    x['Income_at_least_170k'] = income.ge(170_000).astype(np.float32)
    x['Commute_at_least_80'] = commute.ge(80.0).astype(np.float32)

    # Exact repeated values are a feature of this synthetic dataset, not measurement noise.
    for col in RAW_FEATURES:
        x[f'{col}_frequency'] = frame[col].map(frequency_maps[col]).fillna(0.0).astype(np.float32)

    # Digit decompositions expose modulo patterns that ordinary threshold splits cannot learn cheaply.
    categorical = frame[CATEGORICAL_COLS].copy()
    categorical['Subsidy_x_anxiety'] = frame['Subsidy_Available'] + '__' + frame['Range_Anxiety_Level']
    categorical['Subsidy_x_concern'] = frame['Subsidy_Available'] + '__' + concern.astype(int).astype(str)
    categorical['City_x_home_charge'] = frame['City_Type'] + '__' + frame['Home_Charging_Possible']
    categorical['City_x_car_type'] = frame['City_Type'] + '__' + frame['Current_Car_Type']

    income_integer = np.rint(income.to_numpy()).astype(np.int64)
    for place, label in ((1, 'ones'), (10, 'tens'), (100, 'hundreds'), (1_000, 'thousands'), (10_000, 'ten_thousands'), (100_000, 'hundred_thousands')):
        categorical[f'Income_digit_{label}'] = ((income_integer // place) % 10).astype(str)
    commute_tenths = np.rint(commute.to_numpy(dtype=np.float64) * 10.0).astype(np.int64)
    categorical['Commute_tenths_digit'] = (commute_tenths % 10).astype(str)
    categorical['Commute_ones_digit'] = ((commute_tenths // 10) % 10).astype(str)
    categorical['Commute_tens_digit'] = ((commute_tenths // 100) % 10).astype(str)
    age_integer = frame['Age'].to_numpy(dtype=np.int64)
    categorical['Age_ones_digit'] = (age_integer % 10).astype(str)

    one_hot = pd.get_dummies(categorical, prefix=categorical.columns, dtype=np.int8)
    x = pd.concat([x, one_hot], axis=1)
    x.columns = sanitize_feature_names(x.columns.astype(str).tolist())
    x = x.astype(np.float32)
    assert np.isfinite(x.to_numpy()).all(), 'Feature engineering produced NaN or infinity.'
    return x


all_base_features = build_base_features(all_raw)
BASE_FEATURE_NAMES = all_base_features.columns.tolist()
X_TRAIN_BASE = np.ascontiguousarray(all_base_features.iloc[: len(train)].to_numpy(dtype=np.float32))
X_TEST_BASE = np.ascontiguousarray(all_base_features.iloc[len(train):].to_numpy(dtype=np.float32))
del all_base_features, all_raw
gc.collect()

print(f'Base engineered matrix: {X_TRAIN_BASE.shape[1]} features')
print(f'Train matrix memory: {X_TRAIN_BASE.nbytes / 2**20:.1f} MiB')


In [ ]:
FEATURE_GROUPS = {
    'raw_numeric': list(NUMERIC_COLS),
    'binary_ordinal': ['Subsidy_flag', 'Home_charging_flag', 'Range_anxiety_ordinal'],
    'charging': [
        'Total_charging_stations', 'Station_difference', 'Station_minimum',
        'Station_maximum', 'Station_product', 'No_nearby_station',
        'Either_station_zero', 'Home_access_score',
        'Commute_per_station', 'Stations_per_commute',
    ],
    'ratios': ['Income_per_commute', 'Income_per_car'],
    'logs': ['Log_income', 'Log_commute'],
    'numeric_interactions': [
        'Income_x_subsidy', 'Concern_x_subsidy',
        'Concern_minus_anxiety', 'Commute_x_anxiety',
    ],
    'recipe_clues': ['EV_recipe_score', 'EV_recipe_distance', 'EV_recipe_probability'],
    'boundaries': [
        'Annual_Income_USD_winsorized', 'Daily_Commute_km_winsorized',
        'Income_at_floor', 'Commute_at_floor',
        'Income_at_least_170k', 'Commute_at_least_80',
    ],
    'frequency': [column for column in BASE_FEATURE_NAMES if column.endswith('_frequency')],
    'digits': [
        column for column in BASE_FEATURE_NAMES
        if column.startswith(('Income_digit_', 'Commute_tenths_digit_',
                              'Commute_ones_digit_', 'Commute_tens_digit_', 'Age_ones_digit_'))
    ],
    'category_interactions': [
        column for column in BASE_FEATURE_NAMES
        if column.startswith(('Subsidy_x_anxiety_', 'Subsidy_x_concern_',
                              'City_x_home_charge_', 'City_x_car_type_'))
    ],
}
assigned = [column for columns in FEATURE_GROUPS.values() for column in columns]
assert len(assigned) == len(set(assigned)), 'Feature groups overlap.'
assert set(assigned).issubset(BASE_FEATURE_NAMES), 'A registered column does not exist.'
FEATURE_GROUPS['raw_onehot'] = [column for column in BASE_FEATURE_NAMES if column not in set(assigned)]
assert all(column.startswith(tuple(name + '_' for name in CATEGORICAL_COLS))
           for column in FEATURE_GROUPS['raw_onehot'])
all_registered = [column for columns in FEATURE_GROUPS.values() for column in columns]
assert len(all_registered) == len(BASE_FEATURE_NAMES) == len(set(BASE_FEATURE_NAMES))
assert set(all_registered) == set(BASE_FEATURE_NAMES)
display(pd.DataFrame([{'group': name, 'n_features': len(columns)}
                      for name, columns in FEATURE_GROUPS.items()]))


## 3. Cross-fitted target and recipe-residual encodings

Every experiment uses the same inner encoding partitions. An outer training row is encoded from other inner folds; outer validation rows are encoded from outer training labels only. No outer validation label is passed into the encoder.

Exact income/commute encodings expose repeated-value patterns. Residual encodings estimate `target - recipe_probability`. Unseen groups fall back to the fit subset's mean. Optional smoothing experiments change only the common smoothing multiplier.


In [ ]:
# name, grouping columns, smoothing strength, add recipe-residual encoding
TE_SPECS = [
    ('income_exact', ['Annual_Income_USD'], 80.0, True),
    ('commute_exact', ['Daily_Commute_km'], 150.0, True),
    ('income_subsidy_concern', ['Annual_Income_USD', 'Subsidy_Available', 'Environmental_Concern_Level'], 100.0, True),
    ('commute_anxiety', ['Daily_Commute_km', 'Range_Anxiety_Level'], 150.0, True),
    ('subsidy_concern_anxiety', ['Subsidy_Available', 'Environmental_Concern_Level', 'Range_Anxiety_Level'], 500.0, False),
    ('city_home', ['City_Type', 'Home_Charging_Possible'], 500.0, False),
    ('charger_pair', ['Charging_Stations_Near_Home', 'Charging_Stations_Near_Work'], 250.0, False),
    ('age_gender', ['Age', 'Gender'], 250.0, False),
]

TE_FEATURE_NAMES = []
for name, _, _, add_residual in TE_SPECS:
    TE_FEATURE_NAMES.append(f'TE_{name}')
    if add_residual:
        TE_FEATURE_NAMES.append(f'TE_recipe_residual_{name}')
ALL_FEATURE_NAMES = BASE_FEATURE_NAMES + TE_FEATURE_NAMES
RESIDUAL_TE_NAMES = [name for name in TE_FEATURE_NAMES if name.startswith('TE_recipe_residual_')]
STANDARD_TE_NAMES = [name for name in TE_FEATURE_NAMES if name not in RESIDUAL_TE_NAMES]


def smoothed_group_mean(
    fit_frame: pd.DataFrame,
    fit_values: np.ndarray,
    apply_frame: pd.DataFrame,
    columns: list[str],
    smoothing: float,
) -> np.ndarray:
    values = np.asarray(fit_values, dtype=np.float64)
    prior = float(values.mean())
    stats_frame = fit_frame[columns].copy()
    stats_frame['_encoding_value'] = values
    stats = stats_frame.groupby(columns, sort=False, dropna=False)['_encoding_value'].agg(['sum', 'count'])
    encoded_map = (stats['sum'] + smoothing * prior) / (stats['count'] + smoothing)

    if len(columns) == 1:
        encoded = apply_frame[columns[0]].map(encoded_map).to_numpy(dtype=np.float64)
    else:
        apply_index = pd.MultiIndex.from_frame(apply_frame[columns], names=columns)
        encoded = encoded_map.reindex(apply_index).to_numpy(dtype=np.float64)
    return np.nan_to_num(encoded, nan=prior).astype(np.float32)


def build_fold_target_encodings(
    fit_frame: pd.DataFrame,
    fit_y: np.ndarray,
    apply_frames: list[pd.DataFrame],
    seed: int,
    smoothing_multiplier: float = 1.0,
) -> tuple[np.ndarray, list[np.ndarray]]:
    assert smoothing_multiplier > 0
    fit_frame = fit_frame.reset_index(drop=True)
    apply_frames = [frame.reset_index(drop=True) for frame in apply_frames]
    fit_y = np.asarray(fit_y, dtype=np.float64)
    residual_y = fit_y - ev_recipe_probability(fit_frame)

    inner_cv = StratifiedKFold(n_splits=INNER_TE_FOLDS, shuffle=True, random_state=seed)
    inner_splits = list(inner_cv.split(np.zeros(len(fit_y)), fit_y.astype(np.uint8)))
    fit_columns: list[np.ndarray] = []
    apply_columns: list[list[np.ndarray]] = [[] for _ in apply_frames]

    for _, columns, smoothing, add_residual in TE_SPECS:
        smoothing = smoothing * smoothing_multiplier
        value_sets = [fit_y]
        if add_residual:
            value_sets.append(residual_y)

        for values in value_sets:
            cross_fitted = np.empty(len(fit_frame), dtype=np.float32)
            for inner_train_idx, inner_valid_idx in inner_splits:
                cross_fitted[inner_valid_idx] = smoothed_group_mean(
                    fit_frame.iloc[inner_train_idx],
                    values[inner_train_idx],
                    fit_frame.iloc[inner_valid_idx],
                    columns,
                    smoothing,
                )
            fit_columns.append(cross_fitted)

            for apply_number, apply_frame in enumerate(apply_frames):
                apply_columns[apply_number].append(
                    smoothed_group_mean(fit_frame, values, apply_frame, columns, smoothing)
                )

    fit_encoded = np.column_stack(fit_columns).astype(np.float32)
    apply_encoded = [np.column_stack(columns).astype(np.float32) for columns in apply_columns]
    assert fit_encoded.shape[1] == len(TE_FEATURE_NAMES)
    return fit_encoded, apply_encoded


print(f'Target-encoded features per fold: {len(TE_FEATURE_NAMES)}')
print(f'Total model features: {len(ALL_FEATURE_NAMES)}')


## 4. Preflight checks (run before any training)

These small tests check single-column and multi-column encoding, unknown groups, row alignment, finite matrices, and category-safe feature names. They do not fit XGBoost. A failed assertion stops the notebook before the expensive training loop.


In [ ]:
toy_fit = pd.DataFrame({'group': ['a', 'a', 'b', 'b'], 'level': [1, 1, 2, 2]})
toy_values = np.array([0.0, 1.0, 1.0, 1.0])
toy_apply = pd.DataFrame({'group': ['b', 'unknown', 'a'], 'level': [2, 99, 1]},
                         index=[100, 20, 5])
# Global mean = .75; with smoothing 2, a -> .625 and b -> .875.
expected = np.array([0.875, 0.75, 0.625], dtype=np.float32)
assert np.allclose(smoothed_group_mean(toy_fit, toy_values, toy_apply, ['group'], 2), expected)
assert np.allclose(smoothed_group_mean(toy_fit, toy_values, toy_apply, ['group', 'level'], 2), expected)

toy_size = 2 * INNER_TE_FOLDS
toy_raw = train_raw.iloc[:toy_size].reset_index(drop=True)
toy_y = np.tile(np.array([0, 1], dtype=np.uint8), INNER_TE_FOLDS)
toy_encoded, (toy_applied,) = build_fold_target_encodings(
    toy_raw, toy_y, [toy_raw.iloc[:3]], seed=GLOBAL_SEED
)
assert toy_encoded.shape == (toy_size, len(TE_FEATURE_NAMES))
assert toy_applied.shape == (3, len(TE_FEATURE_NAMES))
assert np.isfinite(toy_encoded).all() and np.isfinite(toy_applied).all()
assert len(ALL_FEATURE_NAMES) == len(set(ALL_FEATURE_NAMES))
assert all(not any(character in name for character in '[]<') for name in ALL_FEATURE_NAMES)
assert np.isfinite(X_TRAIN_BASE).all() and np.isfinite(X_TEST_BASE).all()
del toy_fit, toy_values, toy_apply, toy_raw, toy_y, toy_encoded, toy_applied
# Test features supplied only label-free vocabulary/frequency information.
# They are not needed for this training-only study; release their large matrix.
del X_TEST_BASE, test_raw, test
gc.collect()
print('Preflight checks passed.')


## 5. Experiment catalog and fair-comparison checks

The six defaults test the previously proposed ablations. Extra tests can be selected in Section 1:

- `no_recipe_clues`: remove the three recipe feature columns, but retain the prior/residual encodings; tests their additional value.
- `no_recipe_information`: remove **all explicit recipe channels**: feature columns, residual encodings, and starting margin.
- `no_target_residuals`: retain ordinary target encoding but remove recipe-residual encoding.
- `no_interaction_helpers`: remove engineered numeric and one-hot interactions; joint target encodings remain.
- `no_boundaries`, `no_logs`: remove only those companion views.
- `raw_onehot_only`: raw numeric + raw one-hot categories, without prior or target encoding. This is a diagnostic, not a retuned baseline.
- `smoothing_half`, `smoothing_double`: multiply all target-encoding smoothing strengths by 0.5 or 2.
- `prior_half`, `prior_one_and_half`: multiply recipe log-odds by 0.5 or 1.5.

All variants keep `scale_pos_weight=1.0`, as the strongest historical member did. Unlike the previous ensemble-member comparison, the prior on/off pair also has identical model seeds and class weights.

Removing columns changes the number of columns sampled at a fixed sampling fraction; inner/outer splits and all hyperparameters are still controlled. Early stopping follows the same policy for every variant, though the chosen tree count may differ.


In [ ]:
def experiment(drop_groups=(), te_mode='all', prior_scale=1.0, smoothing_scale=1.0, description=''):
    return {
        'drop_groups': list(drop_groups), 'te_mode': te_mode,
        'prior_scale': float(prior_scale), 'smoothing_scale': float(smoothing_scale),
        'description': description,
    }


EXPERIMENT_CATALOG = {
    'full_reference': experiment(description='All features, all TE, recipe prior; strongest single-model family.'),
    'no_digits': experiment(('digits',), description='Remove all income/commute/age digit one-hot features.'),
    'no_frequency': experiment(('frequency',), description='Remove label-free frequency features.'),
    'no_target_encoding': experiment(te_mode='none', description='Remove ordinary and recipe-residual TE.'),
    'no_ratios_charging': experiment(('ratios', 'charging'), description='Remove ratios and charging helpers; raw station counts stay.'),
    'no_recipe_prior': experiment(prior_scale=0.0, description='Disable only the starting margin; all feature clues stay.'),
    'no_recipe_clues': experiment(('recipe_clues',), description='Remove recipe feature columns; prior/residual TE stay.'),
    'no_recipe_information': experiment(('recipe_clues',), te_mode='standard', prior_scale=0.0,
                                        description='Remove recipe columns, residual TE, and prior together.'),
    'no_target_residuals': experiment(te_mode='standard', description='Remove residual TE only.'),
    'no_interaction_helpers': experiment(('numeric_interactions', 'category_interactions'),
                                         description='Remove engineered interactions; joint TE stays.'),
    'no_boundaries': experiment(('boundaries',), description='Remove clipping flags and winsorized companions.'),
    'no_logs': experiment(('logs',), description='Remove log income and commute.'),
    'raw_onehot_only': experiment(
        tuple(group for group in FEATURE_GROUPS if group not in {'raw_numeric', 'raw_onehot'}),
        te_mode='none', prior_scale=0.0, description='Raw numeric and one-hot categories only; frozen settings.'
    ),
    'smoothing_half': experiment(smoothing_scale=0.5, description='Half the target-encoding smoothing.'),
    'smoothing_double': experiment(smoothing_scale=2.0, description='Double the target-encoding smoothing.'),
    'prior_half': experiment(prior_scale=0.5, description='Half the recipe starting log-odds.'),
    'prior_one_and_half': experiment(prior_scale=1.5, description='1.5 times the recipe starting log-odds.'),
}


def feature_selection(spec: dict) -> tuple[np.ndarray, np.ndarray, list[str]]:
    dropped = {column for group in spec['drop_groups'] for column in FEATURE_GROUPS[group]}
    base_indices = np.array([idx for idx, name in enumerate(BASE_FEATURE_NAMES) if name not in dropped],
                            dtype=np.int64)
    retained_te = (
        TE_FEATURE_NAMES if spec['te_mode'] == 'all' else
        STANDARD_TE_NAMES if spec['te_mode'] == 'standard' else []
    )
    te_indices = np.array([idx for idx, name in enumerate(TE_FEATURE_NAMES) if name in retained_te],
                          dtype=np.int64)
    names = [BASE_FEATURE_NAMES[idx] for idx in base_indices] + [TE_FEATURE_NAMES[idx] for idx in te_indices]
    assert names and len(names) == len(set(names))
    return base_indices, te_indices, names


for name in SELECTED_EXPERIMENTS:
    if name not in EXPERIMENT_CATALOG:
        raise ValueError(f'Unknown experiment: {name}. Available: {list(EXPERIMENT_CATALOG)}')
    spec = EXPERIMENT_CATALOG[name]
    assert spec['te_mode'] in {'all', 'standard', 'none'}
    assert spec['prior_scale'] >= 0 and spec['smoothing_scale'] > 0
    assert set(spec['drop_groups']).issubset(FEATURE_GROUPS)
SELECTIONS = {name: feature_selection(spec) for name, spec in EXPERIMENT_CATALOG.items()}

assert SELECTIONS['full_reference'][2] == ALL_FEATURE_NAMES
assert SELECTIONS['no_recipe_prior'][2] == ALL_FEATURE_NAMES
assert len(SELECTIONS['no_target_encoding'][1]) == 0
assert not set(RESIDUAL_TE_NAMES).intersection(SELECTIONS['no_recipe_information'][2])
assert not set(FEATURE_GROUPS['recipe_clues']).intersection(SELECTIONS['no_recipe_information'][2])
assert EXPERIMENT_CATALOG['no_recipe_information']['prior_scale'] == 0.0
for group, name in [('digits', 'no_digits'), ('frequency', 'no_frequency')]:
    assert set(ALL_FEATURE_NAMES) - set(SELECTIONS[name][2]) == set(FEATURE_GROUPS[group])

display(pd.DataFrame([
    {'experiment': name, 'features': len(SELECTIONS[name][2]),
     'removed_features': len(ALL_FEATURE_NAMES) - len(SELECTIONS[name][2]),
     'prior_scale': EXPERIMENT_CATALOG[name]['prior_scale'],
     'smoothing_scale': EXPERIMENT_CATALOG[name]['smoothing_scale'],
     'description': EXPERIMENT_CATALOG[name]['description']}
    for name in SELECTED_EXPERIMENTS
]))


## 6. Isolated outputs and resumable checkpoints

Files go only to `XGBoost/ablation/outputs/<mode>_<fingerprint>/`. Inputs and the frozen parameter snapshot are read-only.

A checkpoint contains validation row indices, predictions, metrics, and its configuration fingerprint. It is reused only if indices, fingerprint, and prediction shape match. Each completed fit is saved immediately using an atomic rename. No partially written checkpoint is accepted. To repeat all fits, set `RESUME_COMPLETED_FOLDS=False`; prior inputs and successful-model folders are never modified.


In [ ]:
def file_sha256(filename: Path) -> str:
    digest = hashlib.sha256()
    with filename.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json(value) -> str:
    return json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)


with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    saved_notebook = json.load(handle)
saved_code = '\n\n'.join(
    ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
    for cell in saved_notebook['cells'] if cell['cell_type'] == 'code'
)
RUN_CONFIG = {
    'pipeline_version': PIPELINE_VERSION, 'mode': MODE, 'global_seed': GLOBAL_SEED,
    'n_splits': N_SPLITS, 'split_seeds': list(SPLIT_SEEDS),
    'inner_te_folds': INNER_TE_FOLDS, 'max_boost_rounds': MAX_BOOST_ROUNDS,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS, 'max_bin': MAX_BIN,
    'device': DEVICE, 'nthread': NTHREAD, 'versions': VERSIONS,
    'reference_search_params': REFERENCE_SEARCH_PARAMS,
    'experiments': {name: EXPERIMENT_CATALOG[name] for name in SELECTED_EXPERIMENTS},
    'feature_groups': FEATURE_GROUPS, 'te_specs': TE_SPECS,
    'data_sha256': {'train': file_sha256(TRAIN_PATH), 'test': file_sha256(TEST_PATH)},
    'snapshot_sha256': file_sha256(SNAPSHOT_PATH),
    'notebook_code_sha256': hashlib.sha256(saved_code.encode('utf-8')).hexdigest(),
}
FINGERPRINT = hashlib.sha256(canonical_json(RUN_CONFIG).encode('utf-8')).hexdigest()
RUN_DIR = STUDY_DIR / 'outputs' / f'{MODE}_{FINGERPRINT[:16]}'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = RUN_DIR / 'run_manifest.json'
if manifest_path.exists():
    with manifest_path.open(encoding='utf-8') as handle:
        if canonical_json(json.load(handle)) != canonical_json(RUN_CONFIG):
            raise ValueError('Existing run manifest is incompatible; do not reuse its checkpoints.')
else:
    with manifest_path.open('w', encoding='utf-8') as handle:
        json.dump(RUN_CONFIG, handle, indent=2)
print('Study outputs:', RUN_DIR)
print('Reference historical single-model OOF AUC:',
      reference_snapshot['run_metadata']['member_oof_auc']['recipe_residual'])
print('Do not compare a 3-fold screen score directly with the historical 10-fold blend.')


In [ ]:
def model_params(model_seed: int) -> dict:
    params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'booster': 'gbtree',
        'tree_method': 'hist', 'device': DEVICE, 'max_bin': MAX_BIN,
        'nthread': NTHREAD, 'seed': model_seed, 'verbosity': 0, 'validate_parameters': True,
        **REFERENCE_SEARCH_PARAMS,
    }
    if params['grow_policy'] == 'depthwise':
        params['max_leaves'] = 0
    else:
        params['max_depth'] = 0
    assert params['scale_pos_weight'] == 1.0
    return params


def prepare_matrix(base: np.ndarray, encoded: np.ndarray, base_indices: np.ndarray,
                   te_indices: np.ndarray) -> np.ndarray:
    parts = [base[:, base_indices]]
    if len(te_indices):
        parts.append(encoded[:, te_indices])
    return np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)


def fit_variant(
    spec: dict, base_train: np.ndarray, base_valid: np.ndarray,
    encoded_train: np.ndarray, encoded_valid: np.ndarray, selection,
    train_y: np.ndarray, valid_y: np.ndarray,
    train_margin: np.ndarray, valid_margin: np.ndarray, model_seed: int,
) -> tuple[np.ndarray, dict]:
    base_indices, te_indices, names = selection
    train_features = prepare_matrix(base_train, encoded_train, base_indices, te_indices)
    valid_features = prepare_matrix(base_valid, encoded_valid, base_indices, te_indices)
    prior_scale = spec['prior_scale']
    dtrain = xgb.QuantileDMatrix(
        train_features, label=train_y, feature_names=names, max_bin=MAX_BIN, nthread=NTHREAD,
        base_margin=prior_scale * train_margin if prior_scale > 0 else None,
    )
    dvalid = xgb.QuantileDMatrix(
        valid_features, label=valid_y, feature_names=names, max_bin=MAX_BIN, nthread=NTHREAD,
        ref=dtrain, base_margin=prior_scale * valid_margin if prior_scale > 0 else None,
    )
    del train_features, valid_features
    history = {}
    started = time.perf_counter()
    model = xgb.train(
        params=model_params(model_seed), dtrain=dtrain, num_boost_round=MAX_BOOST_ROUNDS,
        evals=[(dvalid, 'valid')], evals_result=history, verbose_eval=False,
        callbacks=[xgb.callback.EarlyStopping(
            rounds=EARLY_STOPPING_ROUNDS, metric_name='auc', data_name='valid',
            maximize=True, save_best=True,
        )],
    )
    best_rounds = int(model.best_iteration) + 1
    predictions = model.predict(dvalid, iteration_range=(0, best_rounds)).astype(np.float64)
    assert predictions.shape == valid_y.shape and np.isfinite(predictions).all()
    assert np.all((predictions >= 0) & (predictions <= 1))
    rounds_trained = len(history['valid']['auc'])
    metrics = {
        'auc': float(roc_auc_score(valid_y, predictions)),
        'best_rounds': best_rounds, 'rounds_trained': rounds_trained,
        'hit_round_limit': rounds_trained == MAX_BOOST_ROUNDS,
        'n_features': len(names), 'fit_seconds': float(time.perf_counter() - started),
    }
    del model, dtrain, dvalid
    gc.collect()
    return predictions, metrics


def checkpoint_path(repeat_number: int, fold_number: int, name: str) -> Path:
    return CHECKPOINT_DIR / f'repeat_{repeat_number:02d}_fold_{fold_number:02d}_{name}.npz'


def save_checkpoint(filename: Path, valid_indices: np.ndarray, predictions: np.ndarray,
                    metrics: dict) -> None:
    temporary = filename.with_name(filename.name + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(
            handle, valid_indices=valid_indices, predictions=predictions,
            fingerprint=np.array(FINGERPRINT), metrics_json=np.array(canonical_json(metrics)),
        )
    temporary.replace(filename)


def load_checkpoint(filename: Path, valid_indices: np.ndarray) -> tuple[np.ndarray, dict]:
    with np.load(filename, allow_pickle=False) as saved:
        if str(saved['fingerprint'].item()) != FINGERPRINT:
            raise ValueError(f'Checkpoint has a different configuration: {filename}')
        if not np.array_equal(saved['valid_indices'], valid_indices):
            raise ValueError(f'Checkpoint validation indices differ: {filename}')
        predictions = saved['predictions'].astype(np.float64)
        metrics = json.loads(str(saved['metrics_json'].item()))
    if predictions.shape != (len(valid_indices),) or not np.isfinite(predictions).all():
        raise ValueError(f'Checkpoint predictions are invalid: {filename}')
    if not np.all((predictions >= 0) & (predictions <= 1)):
        raise ValueError(f'Checkpoint predictions are outside [0, 1]: {filename}')
    assert metrics['n_features'] > 0 and 0 < metrics['best_rounds'] <= MAX_BOOST_ROUNDS
    return predictions, metrics


## 7. Run the controlled study

**This is the expensive training cell.** The same model seed is passed to every experiment in a fold. The same inner folds are used for all smoothing variants. Encodings are constructed once per fold/multiplier and reused.

A completed checkpoint is loaded before any fold matrices are built. Default experiments share a single set of target encodings. We hold only one outer fold's feature/encoding arrays at a time. Do not run two kernels writing to the same run directory simultaneously.


In [ ]:
n_repeats = len(SPLIT_SEEDS)
OOF_PREDICTIONS = {
    name: np.full((n_repeats, len(train)), np.nan, dtype=np.float64)
    for name in SELECTED_EXPERIMENTS
}
FOLD_ASSIGNMENTS = np.zeros((n_repeats, len(train)), dtype=np.uint16)
FIT_RECORDS = []
planned_fits = n_repeats * N_SPLITS * len(SELECTED_EXPERIMENTS)
completed_count = 0

for repeat_idx, split_seed in enumerate(SPLIT_SEEDS):
    outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed)
    for fold_number, (train_indices, valid_indices) in enumerate(outer_cv.split(X_TRAIN_BASE, y), start=1):
        repeat_number = repeat_idx + 1
        FOLD_ASSIGNMENTS[repeat_idx, valid_indices] = fold_number
        # Confirm's first repeat reproduces the historical recipe member's seed schedule.
        model_seed = GLOBAL_SEED + 303 + 100_000 * repeat_idx + fold_number
        encoding_seed = GLOBAL_SEED + (10_000 if MODE == 'confirm' else 0) + 100_000 * repeat_idx + fold_number
        pending = []
        print(f'\nRepeat {repeat_number}/{n_repeats}, fold {fold_number}/{N_SPLITS}, split seed {split_seed}')

        for name in SELECTED_EXPERIMENTS:
            filename = checkpoint_path(repeat_number, fold_number, name)
            if RESUME_COMPLETED_FOLDS and filename.is_file():
                predictions, metrics = load_checkpoint(filename, valid_indices)
                if metrics['n_features'] != len(SELECTIONS[name][2]):
                    raise ValueError(f'Checkpoint feature count differs: {filename}')
                actual_auc = float(roc_auc_score(y[valid_indices], predictions))
                if not np.isclose(actual_auc, metrics['auc'], atol=1e-12, rtol=0):
                    raise ValueError(f'Checkpoint AUC differs: {filename}')
                OOF_PREDICTIONS[name][repeat_idx, valid_indices] = predictions
                FIT_RECORDS.append({
                    'experiment': name, 'repeat': repeat_number, 'split_seed': split_seed,
                    'fold': fold_number, 'model_seed': model_seed, 'encoding_seed': encoding_seed,
                    'loaded_checkpoint': True, **metrics,
                })
                completed_count += 1
                print(f"  [{completed_count}/{planned_fits}] {name}: resumed AUC={metrics['auc']:.8f}")
            else:
                pending.append(name)

        if not pending:
            pd.DataFrame(FIT_RECORDS).to_csv(RUN_DIR / 'fold_metrics.csv', index=False)
            continue

        outer_train_raw = train_raw.iloc[train_indices]
        outer_valid_raw = train_raw.iloc[valid_indices]
        base_train = X_TRAIN_BASE[train_indices]
        base_valid = X_TRAIN_BASE[valid_indices]
        train_margin = ev_recipe_logit_margin(outer_train_raw)
        valid_margin = ev_recipe_logit_margin(outer_valid_raw)
        encoding_cache = {}

        for name in pending:
            spec = EXPERIMENT_CATALOG[name]
            if len(SELECTIONS[name][1]):
                multiplier = spec['smoothing_scale']
                if multiplier not in encoding_cache:
                    encoded_train, (encoded_valid,) = build_fold_target_encodings(
                        outer_train_raw, y[train_indices], [outer_valid_raw],
                        seed=encoding_seed, smoothing_multiplier=multiplier,
                    )
                    encoding_cache[multiplier] = (encoded_train, encoded_valid)
                    del encoded_train, encoded_valid
                train_te, valid_te = encoding_cache[multiplier]
            else:
                train_te = np.empty((len(train_indices), 0), dtype=np.float32)
                valid_te = np.empty((len(valid_indices), 0), dtype=np.float32)

            predictions, metrics = fit_variant(
                spec, base_train, base_valid, train_te, valid_te, SELECTIONS[name],
                y[train_indices], y[valid_indices], train_margin, valid_margin, model_seed,
            )
            filename = checkpoint_path(repeat_number, fold_number, name)
            save_checkpoint(filename, valid_indices, predictions, metrics)
            OOF_PREDICTIONS[name][repeat_idx, valid_indices] = predictions
            FIT_RECORDS.append({
                'experiment': name, 'repeat': repeat_number, 'split_seed': split_seed,
                'fold': fold_number, 'model_seed': model_seed, 'encoding_seed': encoding_seed,
                'loaded_checkpoint': False, **metrics,
            })
            completed_count += 1
            cap_note = ' [reached tree limit]' if metrics['hit_round_limit'] else ''
            print(f"  [{completed_count}/{planned_fits}] {name}: "
                  f"AUC={metrics['auc']:.8f}, best trees={metrics['best_rounds']}{cap_note}")
            pd.DataFrame(FIT_RECORDS).to_csv(RUN_DIR / 'fold_metrics.csv', index=False)
            del predictions, train_te, valid_te
            gc.collect()

        encoding_cache.clear()
        del encoding_cache, outer_train_raw, outer_valid_raw, base_train, base_valid, train_margin, valid_margin
        gc.collect()

    for name in SELECTED_EXPERIMENTS:
        assert np.isfinite(OOF_PREDICTIONS[name][repeat_idx]).all(), f'Incomplete OOF: {name}'
    repeat_output = pd.DataFrame({
        ID_COL: train[ID_COL].to_numpy(), TARGET: y,
        'fold': FOLD_ASSIGNMENTS[repeat_idx],
    })
    for name in SELECTED_EXPERIMENTS:
        repeat_output[f'oof_{name}'] = OOF_PREDICTIONS[name][repeat_idx]
    repeat_output.to_csv(RUN_DIR / f'oof_repeat_{repeat_idx + 1:02d}.csv', index=False)
    del repeat_output

assert completed_count == planned_fits
print('\nAll selected experiments have complete OOF predictions.')


## 8. Paired comparisons and report

The main comparison is `candidate OOF AUC - full_reference OOF AUC` **within the same repeat**.

- **Positive delta:** this candidate improved the local score. For a removal experiment, the removed feature group may be unnecessary or harmful.
- **Negative delta:** the full reference performed better; the removed feature group may help.
- Fold win counts describe consistency; they are not significance tests because training folds overlap.
- An isolated tiny gain is not enough. Confirm promising candidates on the second split seed and then consider retuning that feature set.
- Feature importance is not used as proof of a group's contribution. No screening winner is automatically submitted.

The repeat-averaged OOF AUC is saved as an additional ensemble diagnostic, not the primary independent-repeat comparison. Early stopping and previously tuned parameters still introduce model-selection uncertainty; this is a development study, not an untouched final evaluation.


In [ ]:
fold_metrics = pd.DataFrame(FIT_RECORDS).sort_values(['repeat', 'fold', 'experiment']).reset_index(drop=True)
assert len(fold_metrics) == planned_fits
assert not fold_metrics.duplicated(['experiment', 'repeat', 'fold']).any()
assert np.all(FOLD_ASSIGNMENTS > 0)

reference_folds = fold_metrics.loc[fold_metrics['experiment'].eq('full_reference'),
                                  ['repeat', 'fold', 'auc']].rename(columns={'auc': 'reference_fold_auc'})
paired_folds = fold_metrics.merge(reference_folds, on=['repeat', 'fold'], how='left', validate='many_to_one')
paired_folds['delta_vs_reference'] = paired_folds['auc'] - paired_folds['reference_fold_auc']
assert paired_folds['reference_fold_auc'].notna().all()
paired_folds.to_csv(RUN_DIR / 'paired_fold_deltas.csv', index=False)

reference_repeat_auc = [
    float(roc_auc_score(y, OOF_PREDICTIONS['full_reference'][repeat_idx]))
    for repeat_idx in range(n_repeats)
]
repeat_rows = []
summary_rows = []
for name in SELECTED_EXPERIMENTS:
    repeat_auc = np.array([
        float(roc_auc_score(y, OOF_PREDICTIONS[name][repeat_idx]))
        for repeat_idx in range(n_repeats)
    ])
    deltas = repeat_auc - np.array(reference_repeat_auc)
    for repeat_idx in range(n_repeats):
        repeat_rows.append({
            'experiment': name, 'repeat': repeat_idx + 1, 'split_seed': SPLIT_SEEDS[repeat_idx],
            'oof_auc': float(repeat_auc[repeat_idx]),
            'reference_oof_auc': reference_repeat_auc[repeat_idx],
            'delta_vs_reference': float(deltas[repeat_idx]),
        })
    details = paired_folds.loc[paired_folds['experiment'].eq(name)]
    summary_rows.append({
        'experiment': name,
        'mean_repeat_oof_auc': float(repeat_auc.mean()),
        'delta_vs_reference': float(deltas.mean()),
        'worst_repeat_delta': float(deltas.min()),
        'best_repeat_delta': float(deltas.max()),
        'mean_fold_delta': float(details['delta_vs_reference'].mean()),
        'fold_delta_std_descriptive': float(details['delta_vs_reference'].std(ddof=1)),
        'folds_better_than_reference': int((details['delta_vs_reference'] > 0).sum()),
        'total_folds': int(len(details)),
        'n_features': len(SELECTIONS[name][2]),
        'mean_best_trees': float(details['best_rounds'].mean()),
        'fits_reaching_round_limit': int(details['hit_round_limit'].sum()),
        'total_fit_minutes': float(details['fit_seconds'].sum() / 60.0),
        'repeat_averaged_oof_auc': float(roc_auc_score(y, OOF_PREDICTIONS[name].mean(axis=0))),
    })

repeat_metrics = pd.DataFrame(repeat_rows)
summary = pd.DataFrame(summary_rows).sort_values('mean_repeat_oof_auc', ascending=False).reset_index(drop=True)
fold_metrics.to_csv(RUN_DIR / 'fold_metrics.csv', index=False)
repeat_metrics.to_csv(RUN_DIR / 'repeat_oof_metrics.csv', index=False)
summary.to_csv(RUN_DIR / 'ablation_summary.csv', index=False)

feature_manifest = {
    name: {
        'retained': SELECTIONS[name][2],
        'removed': [feature for feature in ALL_FEATURE_NAMES if feature not in set(SELECTIONS[name][2])],
        'specification': EXPERIMENT_CATALOG[name],
    } for name in SELECTED_EXPERIMENTS
}
with (RUN_DIR / 'experiment_features.json').open('w', encoding='utf-8') as handle:
    json.dump(feature_manifest, handle, indent=2)
with (RUN_DIR / 'ablation_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({
        'fingerprint': FINGERPRINT,
        'comparison': 'Paired OOF deltas against the full single-model reference; positive is better.',
        'summary': summary.to_dict(orient='records'),
        'repeat_metrics': repeat_metrics.to_dict(orient='records'),
        'not_an_untouched_holdout': True,
    }, handle, indent=2, allow_nan=False)

display(summary[[
    'experiment', 'mean_repeat_oof_auc', 'delta_vs_reference',
    'worst_repeat_delta', 'folds_better_than_reference', 'total_folds',
    'n_features', 'fits_reaching_round_limit',
]].round(8))
display(repeat_metrics.round(8))
if summary['fits_reaching_round_limit'].sum():
    print('WARNING: Some fits reached MAX_BOOST_ROUNDS. Consider a larger shared budget before concluding.')
if MODE == 'screen':
    print('Next: select full_reference plus promising candidates, set MODE to confirm, and rerun all cells.')
else:
    print('Look for positive deltas across BOTH split seeds before changing the successful pipeline.')
print('Complete results:', RUN_DIR / 'ablation_summary.csv')
print('No test submission was created and no historical-model artifact was modified.')


## What to share after running

Send `ablation_summary.csv` and `repeat_oof_metrics.csv`, or paste the displayed table. These tell us which feature removals helped, which hurt, and whether the improvement repeats.

For confirmation, change `MODE='confirm'` and keep `full_reference` plus only a few promising candidates in `SELECTED_EXPERIMENTS`. The first confirmation seed reproduces the historical recipe model's fold/encoding/model-seed schedule when defaults are unchanged. Confirm's second seed tests robustness. The reference should be near the historical **single-model** OOF AUC 0.945722, not the leaderboard score or ensemble AUC.

Use the optional smoothing/prior variants to investigate the strongest components after the first ablations. Interacting feature families can mask each other's effects; `no_recipe_information` specifically removes all explicit recipe-derived channels together.
